# Step 2. 분석 대상 게임 리스트 확보

**목표**: 5개 장르 × 10개 = 총 50개 게임 선정  
**선정 기준**: 유료 게임(Free-to-Play 제외) / 정가 $5 이상 / 리뷰 10,000개 이상 / 출시 1년 이상 경과  
**출력**: `data/target_games.csv`

### 데이터 소스
- **SteamSpy API** (무료/키 불필요) : 장르별 게임 목록 + 리뷰 수
- **Steam Store API** (무료/키 불필요) : 출시일 / 정가 / F2P 여부 확인

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from dateutil import parser as dateutil_parser
from dotenv import load_dotenv
from tqdm.auto import tqdm
import os

load_dotenv()
STEAM_API_KEY = os.getenv("STEAM_API_KEY")
print("STEAM_API_KEY:", "OK" if STEAM_API_KEY else "MISSING (.env 확인 필요)")

## 설정값

In [1]:
MIN_REVIEWS = 10_000          # 최소 리뷰 수
MIN_PRICE_USD = 5.0           # 최소 정가 (USD) — F2P 및 초저가 게임 제외
MIN_RELEASE_AGO_DAYS = 365    # 출시 후 최소 경과 일수
TARGET_PER_GENRE = 10         # 장르당 선정 수
TODAY = datetime.now()
CUTOFF_DATE = TODAY - timedelta(days=MIN_RELEASE_AGO_DAYS)

# 장르 키 → SteamSpy 태그 매핑
GENRE_CONFIG = {
    "Action":              ["Action"],
    "RPG":                 ["RPG"],
    "Strategy_Simulation": ["Strategy", "Simulation"],
    "Adventure":           ["Adventure"],
    "Casual_Indie":        ["Indie"],
}

GENRE_LABEL = {
    "Action":              "Action",
    "RPG":                 "RPG",
    "Strategy_Simulation": "Strategy/Simulation",
    "Adventure":           "Adventure",
    "Casual_Indie":        "Casual/Indie",
}

print(f"선정 기준: 유료 게임 / 정가 ${MIN_PRICE_USD}+ / 리뷰 {MIN_REVIEWS:,}개+ / 출시 {MIN_RELEASE_AGO_DAYS}일+")
print(f"출시 컷오프: {CUTOFF_DATE.strftime('%Y-%m-%d')}")

NameError: name 'datetime' is not defined

## 헬퍼 함수

In [ ]:
def fetch_steamspy_genre(genre_tag):
    """SteamSpy에서 특정 장르 게임 목록 가져오기 (최대 1000개)"""
    url = f"https://steamspy.com/api.php?request=genre&genre={genre_tag}"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    games = []
    for appid, info in data.items():
        total_reviews = info.get("positive", 0) + info.get("negative", 0)
        games.append({
            "appid": int(appid),
            "name": info.get("name", ""),
            "positive": info.get("positive", 0),
            "negative": info.get("negative", 0),
            "total_reviews": total_reviews,
        })
    return games


def get_app_info(appid):
    """
    Steam store API에서 출시일 / 정가 / F2P 여부를 한 번에 가져오기.
    반환: {release_dt, price_usd, is_f2p} 또는 None
    """
    url = (
        f"https://store.steampowered.com/api/appdetails"
        f"?appids={appid}&cc=us&l=en&filters=release_date,price_overview,genres"
    )
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json().get(str(appid), {})
        if not data.get("success"):
            return None
        d = data["data"]

        # 출시일
        release = d.get("release_date", {})
        if release.get("coming_soon"):
            return None
        date_str = release.get("date", "").strip()
        if not date_str:
            return None
        release_dt = dateutil_parser.parse(date_str)

        # 정가 (cents → dollars)
        price_info = d.get("price_overview", {})
        price_usd = price_info.get("initial", 0) / 100.0

        # F2P 여부: 장르에 "Free to Play" 포함 여부
        genre_list = [g["description"] for g in d.get("genres", [])]
        is_f2p = "Free to Play" in genre_list

        return {"release_dt": release_dt, "price_usd": price_usd, "is_f2p": is_f2p}
    except Exception:
        return None

print("함수 정의 완료")

## Step 2-1. SteamSpy에서 장르별 후보 수집

리뷰 10,000개 이상으로 1차 필터링 후 리뷰 수 내림차순 정렬.

In [ ]:
candidates = {}

for genre_key, steamspy_tags in GENRE_CONFIG.items():
    label = GENRE_LABEL[genre_key]
    genre_games = []

    for tag in steamspy_tags:
        print(f"  SteamSpy 조회 중: {tag}...")
        games = fetch_steamspy_genre(tag)
        genre_games.extend(games)
        time.sleep(2)

    # appid 기준 중복 제거
    seen = set()
    unique = []
    for g in genre_games:
        if g["appid"] not in seen:
            seen.add(g["appid"])
            unique.append(g)

    # 리뷰 수 필터 + 내림차순 정렬
    filtered = [g for g in unique if g["total_reviews"] >= MIN_REVIEWS]
    filtered.sort(key=lambda x: x["total_reviews"], reverse=True)

    candidates[genre_key] = filtered
    print(f"[{label}] 후보 {len(filtered)}개 확보\n")

print("=" * 40)
for k, v in candidates.items():
    print(f"  {GENRE_LABEL[k]:<25} {len(v):>4}개")

## Step 2-2. Steam API로 검증 → 최종 50개 선정

리뷰 수 상위 후보부터 Steam API 호출.  
**F2P 제외 / 정가 $5 미만 제외 / 출시 1년 미만 제외**  
장르당 10개 확보되면 즉시 다음 장르로 이동.

In [ ]:
already_selected = set()
selected_games = []

for genre_key, game_list in candidates.items():
    label = GENRE_LABEL[genre_key]
    genre_selected = []
    skipped_new = 0
    skipped_f2p = 0
    skipped_price = 0
    skipped_err = 0

    print(f"\n[{label}] 검증 시작...")

    for game in tqdm(game_list, desc=label):
        if len(genre_selected) >= TARGET_PER_GENRE:
            break
        if game["appid"] in already_selected:
            continue

        info = get_app_info(game["appid"])
        time.sleep(1.5)

        if info is None:
            skipped_err += 1
            continue
        if info["is_f2p"] or info["price_usd"] == 0:
            skipped_f2p += 1
            continue
        if info["price_usd"] < MIN_PRICE_USD:
            skipped_price += 1
            continue
        if info["release_dt"] > CUTOFF_DATE:
            skipped_new += 1
            continue

        game["genre_category"] = label
        game["release_date"] = info["release_dt"].strftime("%Y-%m-%d")
        genre_selected.append(game)
        already_selected.add(game["appid"])

    print(f"  선정: {len(genre_selected)}개 | F2P 제외: {skipped_f2p}개 | 신작 제외: {skipped_new}개 | 저가 제외: {skipped_price}개 | 실패: {skipped_err}개")

    if len(genre_selected) < TARGET_PER_GENRE:
        print(f"  ⚠️  목표({TARGET_PER_GENRE}개)보다 {TARGET_PER_GENRE - len(genre_selected)}개 부족.")

    selected_games.extend(genre_selected)

print(f"\n{'='*40}")
print(f"총 선정: {len(selected_games)}개")

## 결과 확인

In [ ]:
df = pd.DataFrame(selected_games)[[
    "appid", "name", "genre_category", "release_date",
    "total_reviews", "positive", "negative"
]]

print("장르별 선정 결과:")
print(df.groupby("genre_category")["appid"].count().rename("게임 수"))
print(f"\n합계: {len(df)}개")
print(f"리뷰 수 범위: {df['total_reviews'].min():,} ~ {df['total_reviews'].max():,}")
df

## CSV 저장

In [ ]:
output_path = "../data/target_games.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"Shape: {df.shape}")